# File 2nd - Random gradients selection analysis
# Gradient/bval signal ANALYSIS for FF_dwi_drift
------------------------------------------------------

1) The first idea is to identify signal drift in the acuqired signal - In order to do it, we will run several GLM analysis in each gradient, hopefully we will find significant effect of gradients on spacial signal. 
b-value: the .bval file gives the overall diffusion weighting for that volume, from the Stejskal-Tanner relation: b = γ²G²δ²(Δ − δ/3) where G is gradient amplitude, δ is pulse duration, and Δ is the time between the two diffusion gradient lobes.
    b = γ²G²δ²(Δ − δ/3): [i asked Claude]
        
        γ = 2.675×10⁸ rad/s/T (water protons)
        δ (pulse duration)	~20 ms	
        Δ (lobe separation)	~35 ms	


2) Separate each bval --> Average volume * each bval then compute a GLM between ses-01 and ses-02 on the merged sequence. 

In [ ]:
import re
import numpy as np
import os
import nibabel as nib
from nilearn import image as nlimage
import tempfile

vps = [i for i in range(44) if i not in [32]]
sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]
SMOOTH_FWHM_MM = "6"
TEMPLATE = "MNI"

pid = "sub-00"
ses = "ses-01"
subjid = f"{pid}_{ses}"

home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

AP = "A"
dvs_path = f"/home/malberti/wks14/temp/FF_DWI_Drift/DWI_plit_{AP}.dvs"
bvals = os.path.join(home, pid, ses, "dwi", "signal_drift", f"{subjid}_dwi_eddy_corrected_{AP}_noPA.bval")

BVALs = np.loadtxt(bvals)
#BVALs = BVALs[1::]  # Remove the first b0

vectors = []
with open(dvs_path) as f:
    for line in f:
        match = re.match(r"Vector\[\d+\]\s*=\s*\(([-\d.]+),([-\d.]+),([-\d.]+)\)", line)
        if match:
            vectors.append([float(match.group(1)), float(match.group(2)), float(match.group(3))])

vectors = np.array(vectors)
print(f"[INFO] Parsed {len(vectors)} | {len(BVALs)} vectors from {dvs_path}")

if len(vectors) == len(BVALs) - 1:  # The bvals have one additional b=0 from the scanner
    vectors = np.vstack([[0.0, 0.0, 0.0], vectors])
    print(f"[INFO] Padded vectors: {len(vectors)} == {len(BVALs)}")
elif len(vectors) != len(BVALs):
    raise ValueError(f"Unexpected mismatch: vectors={len(vectors)}, BVALs={len(BVALs)}")

# columns: x, y, z, bval, original_volume_index
indices = np.arange(len(BVALs))
VECTORs = np.column_stack([vectors, BVALs, indices])
tolerance = 50

SHELLs = np.unique(np.round(BVALs / tolerance) * tolerance)

# compute octant-pair volume indices per shell ONCE, before touching any participant data
shell_octant_indices = {}  # {shell: {"FIRST_Oct": [orig_vol_idx, ...], ...}}

for shell in SHELLs:
    bSHELL_Vectors = VECTORs[(VECTORs[:, 3] > shell - tolerance) & (VECTORs[:, 3] < shell + tolerance)]

    FIRST_Oct = bSHELL_Vectors[
        ((bSHELL_Vectors[:, 0] > 0) & (bSHELL_Vectors[:, 1] < 0) & (bSHELL_Vectors[:, 2] > 0)) |
        ((bSHELL_Vectors[:, 0] < 0) & (bSHELL_Vectors[:, 1] > 0) & (bSHELL_Vectors[:, 2] < 0))
    ]
    SECOND_Oct = bSHELL_Vectors[
        ((bSHELL_Vectors[:, 0] > 0) & (bSHELL_Vectors[:, 1] < 0) & (bSHELL_Vectors[:, 2] < 0)) |
        ((bSHELL_Vectors[:, 0] < 0) & (bSHELL_Vectors[:, 1] > 0) & (bSHELL_Vectors[:, 2] > 0))
    ]
    THIRD_Oct = bSHELL_Vectors[
        ((bSHELL_Vectors[:, 0] > 0) & (bSHELL_Vectors[:, 1] > 0) & (bSHELL_Vectors[:, 2] > 0)) |
        ((bSHELL_Vectors[:, 0] < 0) & (bSHELL_Vectors[:, 1] < 0) & (bSHELL_Vectors[:, 2] < 0))
    ]
    FOURTH_Oct = bSHELL_Vectors[
        ((bSHELL_Vectors[:, 0] > 0) & (bSHELL_Vectors[:, 1] > 0) & (bSHELL_Vectors[:, 2] < 0)) |
        ((bSHELL_Vectors[:, 0] < 0) & (bSHELL_Vectors[:, 1] < 0) & (bSHELL_Vectors[:, 2] > 0))
    ]

    shell_octant_indices[shell] = {
        "FIRST_Oct": FIRST_Oct[:, 4].astype(int),
        "SECOND_Oct": SECOND_Oct[:, 4].astype(int),
        "THIRD_Oct": THIRD_Oct[:, 4].astype(int),
        "FOURTH_Oct": FOURTH_Oct[:, 4].astype(int),
    }
    print(f"[INFO] shell {shell}: FIRST={len(FIRST_Oct)} SECOND={len(SECOND_Oct)} THIRD={len(THIRD_Oct)} FOURTH={len(FOURTH_Oct)}")

recap_lines = ["# Volume / Index / Bval correspondence\n", f"AP direction: {AP}\n"]

for shell, octants in shell_octant_indices.items():
    recap_lines.append(f"\n## Shell b{int(shell)}\n")
    for label, vol_indices in octants.items():
        recap_lines.append(f"\n### {label}\n")
        recap_lines.append("| volume_index | bval | x | y | z |\n")
        recap_lines.append("|---|---|---|---|---|\n")
        for idx in vol_indices:
            x, y, z = vectors[idx]
            recap_lines.append(f"| {idx} | {BVALs[idx]:.1f} | {x:.4f} | {y:.4f} | {z:.4f} |\n")

recap_content = "".join(recap_lines)

# now average DWI volumes per participant / session / shell / octant
out_home = r"/home/malberti/Unix_Folders/SWEEP2/Gradients_Stability_Analysis"
out_dir = os.path.join(out_home, "Gradients_Stability_Quadrants")
os.makedirs(out_dir, exist_ok=True)


for vp in vps:
    for session in sessions:
        pid = f"sub-{vp:02d}"
        subjid = f"{pid}_{session}"

        dwi_path = os.path.join(out_home, "Volume_MNI", f"{subjid}_{AP}_desc-MNI.nii.gz")
        dwi_img = nib.load(dwi_path)

        with tempfile.TemporaryDirectory() as tmp:
            smoothed = nlimage.smooth_img(dwi_img, fwhm=float(SMOOTH_FWHM_MM))
            dwi_out = os.path.join(tmp, f"{subjid}_{AP}_desc-MNI{SMOOTH_FWHM_MM}mm.nii.gz")
            smoothed.to_filename(dwi_out)
            dwi_img = nib.load(dwi_out)

            dwi_affine = dwi_img.affine
            dwi_header = dwi_img.header
            dwi_data = dwi_img.get_fdata()  # load fully while the tmp file still exists

            recap_path = os.path.join(out_dir, f"{subjid}_{AP}_recap_readme.md")
            with open(recap_path, "w") as f:
                f.write(recap_content)
            print(f"[INFO] Saved {recap_path}")

            for shell, octants in shell_octant_indices.items():
                quadrant_avgs = []
                labels_order = []

                for label, vol_indices in octants.items():
                    quadrant_avg = np.mean(dwi_data[..., vol_indices], axis=3)
                    quadrant_avgs.append(quadrant_avg)
                    labels_order.append(label)
                    print(f"[INFO] shell {int(shell)} {label}: n_vols={len(vol_indices)}")

                stacked = np.stack(quadrant_avgs, axis=3)  # 4D: (x, y, z, 4) — one volume per octant, in labels_order

                out_path = os.path.join(out_dir, f"{subjid}_{AP}_b{int(shell)}_desc-MNI{SMOOTH_FWHM_MM}mm.nii.gz")
                nib.Nifti1Image(stacked, dwi_affine, dwi_header).to_filename(out_path)
                print(f"[INFO] Saved {out_path}  shape={stacked.shape}  order={labels_order}")

print("[INFO] Done")

# Run paired t-test(GLM)
--------------------------------

#Old plot -----------------

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

octant_groups = {
    "FIRST_Oct": (FIRST_Oct, "blue"),
    "SECOND_Oct": (SECOND_Oct, "orange"),
    "THIRD_Oct": (THIRD_Oct, "green"),
    "FOURTH_Oct": (FOURTH_Oct, "red"),
}

for label, (data, color) in octant_groups.items():
    fig.add_trace(go.Scatter3d(
        x=data[:, 0], y=data[:, 1], z=data[:, 2],
        mode="markers", marker=dict(size=4, color=color),
        name=label,
    ))

fig.update_layout(scene=dict(aspectmode="cube"), title=f"b{shell} directions by octant-pair")
fig.show()

# Run the same shit but on the merged sequence in order to increase the sampling points
----------------------------------


In [ ]:
## Same code but in parallel (using joblib) to speed up the processing of multiple subjects/sessions.

import os
import glob
import subprocess
import tempfile
import numpy as np
import nibabel as nib
from nilearn import image as nlimage
from concurrent.futures import ProcessPoolExecutor
import warnings

warnings.filterwarnings("ignore")

# ===============================================
# Configuration
# ===============================================

vps      = [i for i in range(44) if i not in [32]]   # list of subject IDs
sessions = ["ses-01", "ses-02"]

home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

TEMPLATE = "MNI"   # "HCPex" or "MNI"
TEMPLATE_PATHS = {
    "HCPex": r"/home/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/Atlases/MNI_icbm_152_Template_pl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz",
    "MNI":   r"/usr/local/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz",
}

TEMPLATE_PATH = TEMPLATE_PATHS[TEMPLATE]

BVAL_TOLERANCE = 50
SMOOTH_FWHM_MM = 6
N_WORKERS = 10


def process_subject_session(vp, ses):

    pid = f"sub-{vp:02d}"
    subjid = f"{pid}_{ses}"
    derivatives = os.path.join(home, pid, ses, "dwi")

    DWIs = os.path.join(derivatives, "signal_drift", f"{subjid}_eddy-current_signal-drift_ABC-2T1w_corr.nii.gz")
    BVALs = np.loadtxt(os.path.join(derivatives, "eddy", f"{subjid}_dwi_eddy_corrected_ABC_noPA2T1w.bval"))
    SHELLs = np.unique(np.round(BVALs / BVAL_TOLERANCE) * BVAL_TOLERANCE)

    anat = os.path.join(home, pid, "ses-02", "anat")

    t1w2MNI = os.path.join(anat, "mni-registration", f"{pid}_ses-02_desc-nonlin1warp_xfm.nii.gz")
    t1w2MNI_lin = os.path.join(anat, "mni-registration", f"{pid}_ses-02_desc-nonlin0genericaffine_xfm.mat")

    # Outdir
    out_dir = os.path.join(derivatives, "Signal_Stability")
    os.makedirs(out_dir, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmp:
        print(f"Splitting: {DWIs}")
        subprocess.run(f"fslsplit {DWIs} {os.path.join(tmp, 'vol')}", shell=True, check=True)

        files = sorted(glob.glob(os.path.join(tmp, "vol*")))

        for i, file in enumerate(files):

            print(f"[INFO] Coregistering {file} | {t1w2MNI} and {t1w2MNI_lin}")

            out_path = os.path.join(out_dir,f"{subjid}_ABC_vol{i:03d}_2MNI6mm.nii.gz")

            cmd = [
                "antsApplyTransforms", "-d", "3",
                "-i", file,
                "-r", TEMPLATE_PATH,
                "-o", out_path,
                "-t", t1w2MNI,
                "-t", t1w2MNI_lin,
                "--interpolation", "Linear",
            ]

            subprocess.run(cmd, check=True)

            # Smooths
            smoothed = nlimage.smooth_img(out_path, fwhm=SMOOTH_FWHM_MM)
            smoothed.to_filename(out_path)

            print(f"[INFO] Saved {out_path}")

    return out_dir

# ===============================================
# STAGE 1 — per subject/session: extract shells, average directions, coreg
# ===============================================

if __name__ == "__main__":
    jobs = [(vp, ses) for vp in vps for ses in sessions]
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = {ex.submit(process_subject_session, vp, ses): (vp, ses) for vp, ses in jobs}
        for f in futures:
            vp, ses = futures[f]
            try:
                f.result()
            except Exception as e:
                print(f"[ERROR] sub-{vp:02d} {ses} failed: {e}")

In [ ]:

import os
import glob
import subprocess
import tempfile
import numpy as np
import nibabel as nib
from nilearn import image as nlimage
from concurrent.futures import ProcessPoolExecutor
import warnings
import re
import pandas as pd
from nilearn.glm import threshold_stats_img
from nilearn.plotting import plot_design_matrix, plot_glass_brain
import matplotlib.pyplot as plt
from nilearn.glm.second_level import SecondLevelModel

warnings.filterwarnings("ignore")

# ===============================================
# Configuration
# ===============================================

vps      = [i for i in range(44) if i not in [32]]   # list of subject IDs
sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]
home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

TEMPLATE = "MNI"   # "HCPex" or "MNI"
TEMPLATE_PATHS = {
    "HCPex": r"/home/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/Atlases/MNI_icbm_152_Template_pl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz",
    "MNI":   r"/usr/local/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz",
}

TEMPLATE_PATH = TEMPLATE_PATHS[TEMPLATE]

BVAL_TOLERANCE = 50
SMOOTH_FWHM_MM = 6
N_WORKERS = 10

In [ ]:
QUADRANT_indx = {}  # {(AP, shell): {"FIRST_Oct": [...], ...}}
VECTORs = []
VECTOR = []
APs = ["A", "B", "C"]
for AP in APs:
    dvs_path = f"/home/malberti/wks14/temp/FF_DWI_Drift/DWI_plit_{AP}.dvs"
    bvals = os.path.join(home, "sub-04", "ses-01", "dwi", "signal_drift", f"sub-04_ses-01_dwi_eddy_corrected_{AP}_noPA.bval")
    BVALs = np.loadtxt(bvals)

    vectors = []
    with open(dvs_path) as f:
        for line in f:
            match = re.match(r"Vector\[\d+\]\s*=\s*\(([-\d.]+),([-\d.]+),([-\d.]+)\)", line)
            if match:
                vectors.append([float(match.group(1)), float(match.group(2)), float(match.group(3))])

    vectors = np.array(vectors)
    vectors = np.vstack([[0.0, 0.0, 0.0], vectors])
    AP_marks = np.full((len(vectors), 1), AP)

    print(f"[INFO] AP={AP}: parsed {len(vectors)} | {len(BVALs)} vectors from {dvs_path}")

    VECTOR.append(np.column_stack((vectors, BVALs, AP_marks)))

index = np.tile(np.arange(121), 3).reshape(-1, 1)             # collapse the list of (n_i, 4) arrays into one (total_n, 4) array
VECTORs = np.vstack(VECTOR)              # collapse the list of (n_i, 4) arrays into one (total_n, 4) array
VECTORs = np.column_stack([VECTORs, index])  # now (total_n, 5): x, y, z, bval, index


### Sample only 32 dir. * shell in order to balance the b500 
-------------------------------------------------------------------


In [ ]:
tolerance = 50
SHELLs = np.unique(np.round(BVALs / tolerance) * tolerance)
N_volumes = 0
MAX_PER_OCT = 8

for shell in SHELLs:
    if shell > 800:
        bSHELL_Vectors = VECTORs[(VECTORs[:, 3].astype(float) > shell - tolerance) & (VECTORs[:, 3].astype(float) < shell + tolerance)]

        FIRST_Oct = bSHELL_Vectors[
            ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) > 0)) |
            ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) < 0))
        ]
   
        SECOND_Oct = bSHELL_Vectors[
            ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) < 0)) |
            ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) > 0))
        ]
        THIRD_Oct = bSHELL_Vectors[
            ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) > 0)) |
            ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) < 0))
        ]
        FOURTH_Oct = bSHELL_Vectors[
            ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) < 0)) |
            ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) > 0))
        ]

        if shell > 1100:
            QUADRANT_indx[shell] = {
                "FIRST_Oct": FIRST_Oct[np.random.choice(len(FIRST_Oct), min(MAX_PER_OCT, len(FIRST_Oct)), replace=False)],
                "SECOND_Oct": SECOND_Oct[np.random.choice(len(SECOND_Oct), min(MAX_PER_OCT, len(SECOND_Oct)), replace=False)],
                "THIRD_Oct": THIRD_Oct[np.random.choice(len(THIRD_Oct), min(MAX_PER_OCT, len(THIRD_Oct)), replace=False)],
                "FOURTH_Oct": FOURTH_Oct[np.random.choice(len(FOURTH_Oct), min(MAX_PER_OCT, len(FOURTH_Oct)), replace=False)],
            }

            QUADRANT_indx[(shell)] = {
                            "FIRST_Oct": FIRST_Oct[:],
                        "SECOND_Oct": SECOND_Oct[:],
                        "THIRD_Oct": THIRD_Oct[:],
                        "FOURTH_Oct": FOURTH_Oct[:],
                    }


        elif shell < 1100: 
        
            FIRST_Oct_ids = [6, 7, 8, 9, 10, 11, 12, 13] # [0, 1, 2, 3, 4, 5, 6, 7]
            SECOND_Oct_ids = [8, 9, 10, 11, 12, 13, 14, 15]#[0, 1, 2, 3, 4, 5, 6, 7]
            THIRD_Oct_ids = [8, 9, 10, 11, 12, 13, 14, 15] #[0, 1, 2, 3, 4, 5, 6, 7]
            FOURTH_Oct_ids = [8, 9, 10, 11, 12, 13, 14, 15]#[0, 1, 2, 3, 4, 5, 6, 7]

            QUADRANT_indx[shell] = {
                        "FIRST_Oct": FIRST_Oct[FIRST_Oct_ids],
                        "SECOND_Oct": SECOND_Oct[SECOND_Oct_ids],
                        "THIRD_Oct": THIRD_Oct[THIRD_Oct_ids],
                        "FOURTH_Oct": FOURTH_Oct[FOURTH_Oct_ids],
                    }

           
        elif shell < 800: 
            bSHELL_Vectors = VECTORs[(VECTORs[:, 3].astype(float) > shell - tolerance) & (VECTORs[:, 3].astype(float) < shell + tolerance)]

            FIRST_Oct = bSHELL_Vectors[
                ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) > 0)) |
                ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) < 0))
            ]

            SECOND_Oct = bSHELL_Vectors[
                ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) < 0)) |
                ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) > 0))
            ]
            THIRD_Oct = bSHELL_Vectors[
                ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) > 0)) |
                ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) < 0))
            ]
            FOURTH_Oct = bSHELL_Vectors[
                ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) < 0)) |
                ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) > 0))
            ]
        

            QUADRANT_indx[(shell)] = {
                    "FIRST_Oct": FIRST_Oct[:],
                "SECOND_Oct": SECOND_Oct[:],
                "THIRD_Oct": THIRD_Oct[:],
                "FOURTH_Oct": FOURTH_Oct[:],
            }

    N_volumes = N_volumes + len(bSHELL_Vectors)
    print(f"[INFO]  shell {shell}: FIRST={len(FIRST_Oct)} SECOND={len(SECOND_Oct)} THIRD={len(THIRD_Oct)} FOURTH={len(FOURTH_Oct)}")
    print(f"[INFO] shell {shell}: {len(bSHELL_Vectors)} volumes/indices")

print(f"[INFO] Grand total across all shells: {N_volumes}")

In [ ]:

sessions = ["ses-01","ses-02"]
APs = ["A", "B", "C"] # TO merge all together, we need to play around A, B and C 
# If ww consider the A->B->C acquistion order, then the first 120 volumes, correpond to A. 
# so, i the volume is between 0 - 120 --> then it goes directly to A 
# if the volume is between 121 - 241 --> then vol = B (vol_numb - 120)
# if the volume is between 242 - 362 --> then vol = B (vol_numb - 120)
#

home = r"/home/malberti/Unix_Folders/SWEEP2/Gradients_Stability_stacked"
results={}

for shell, quadrant in QUADRANT_indx.items():
    if shell < 900: 
         continue 
    print(f"\n================ SHELL {shell} ================")
    results[shell] = {}
    for label, vol_indices in quadrant.items():
            print(f"Computing: Quadrant = {label}\n")
            imgs_ses01_ses02_quadrant = []
            count = 0 
            for session in sessions: 
                print(f"Stacking: Session = {session}\n")
                imgs = [] #stack all the volumes for one session 

                for vol in vol_indices:
                    AP = vol[4]
                    v = vol[5]
                    DWI = os.path.join(home, f"{session}_{AP}_vol{v}.nii")
                    dwi_img = nib.load(DWI)
                    print(DWI)

                    for i in range(dwi_img.shape[3]):
                        imgs.append(nib.Nifti1Image(dwi_img.slicer[..., i].get_fdata(), dwi_img.affine, dwi_img.header)) 
                        count +=1
                       # print(f"Stacking volume {i}/{dwi_img.shape[3]}")
                
                imgs_ses01_ses02_quadrant = imgs_ses01_ses02_quadrant +imgs
                print(imgs_ses01_ses02_quadrant)     
            
            
            if len(imgs_ses01_ses02_quadrant) % 2 == 0: # Quick Check
                print("OK: even number of images")
                n_subj = int(count/2) #Number of volumes * sessions
                subjects = [f"vol-{vvol:02d}" for vvol in range(n_subj)]
                condition_effect = np.hstack(([1] * n_subj, [-1] * n_subj))  # ses-01 block, then ses-02 block
                subject_effect = np.vstack((np.eye(n_subj), np.eye(n_subj)))
                paired_design_matrix = pd.DataFrame(
                    np.hstack((condition_effect[:, np.newaxis], subject_effect)),
                    columns=["PORCO_vs_DIO"] + subjects, #ses-01 first
                )
                plot_design_matrix(paired_design_matrix)
                plt.show()
            else:
                print("ERROR: odd number of images")
                break 
                        # Check that ses-01 and ses-02 are properly paire 
    
            # --- nilearn GLM: t-test appaiato ---
            model = SecondLevelModel(n_jobs=2, verbose=0).fit(imgs_ses01_ses02_quadrant, design_matrix=paired_design_matrix)
            maps = model.compute_contrast("PORCO_vs_DIO", output_type="all")

        
            thresholded_map, threshold = threshold_stats_img(
                    maps["z_score"], alpha=0.001, cluster_threshold=10, two_sided=True
                )

            results[shell][label] = {"map": thresholded_map, "threshold": threshold}

           # zmap_path = os.path.join(out_dir, f"paired_ttest_zmap_b{int(shell)}_{label}.nii.gz")
          #  thresholded_map.to_filename(zmap_path)
          #  print(f"[INFO] shell {int(shell)} {label}: salvato {zmap_path}  threshold={threshold:.2f}")

            # --- glass brain plot ---
            display = plot_glass_brain(
                thresholded_map,
                threshold=threshold,
                colorbar=True,
                plot_abs=False,
                display_mode="ortho",
                title=f"b{int(shell)} {label} | ses-01 vs ses-02 (z>{threshold:.2f})",
            )


            plt.show()